# Supercomputing Internet LLM Fine-Tuning
## Multi-GPU Distributed Fine-Tuning with FSDP + LoRA

This notebook is an English, GitHub-friendly translation of the original Chinese walkthrough.

It demonstrates how to use cloud notebooks with two 4090 GPUs to fine-tune `deepseek-ai/deepseek-llm-7b-base` using FSDP + LoRA for multi-GPU distributed training.

The Python code blocks are kept intact as requested.

## Notes

This article is an extension of “Supercomputing Internet LLM Fine-Tuning LoRA Example” and focuses on a multi-GPU distributed training example. The registration steps and usage details are referenced from the original article.

In this version, step 7 is modified to use two 4090 accelerator cards to demonstrate multi-GPU distribution.

Because the 7B model has higher generation freedom (DoF), the prompt is adjusted to improve training stability:

```python
instruction = "Analyze the sentiment of the following tweet. Output exactly one word, with no punctuation or extra text:"
```

Readers who are already familiar with the workflow can jump directly to steps 11–17, then to steps 26–30 for the quick-start code path.

After is configured with two 4090 GPUs, switch to the Aliyun PyPI mirror, install the libraries, upload `twitter-airline-sentimentSentiment_Analysis.csv` into the same folder as the `.ipynb` file, and download the `deepseek-ai/deepseek-llm-7b-base` model into `/root/private_data/DeepSeek7B`.

## Background: DDP vs. FSDP

PyTorch distributed training mainly uses two approaches:

- **DDP (Distributed Data Parallel)**: the full model weights are replicated on each GPU and computation is performed in parallel.
- **FSDP (Fully Sharded Data Parallel)**: model weights are split into shards and distributed across GPUs.

This notebook uses **FSDP + LoRA** for multi-GPU fine-tuning.

## Steps 1–10: setup

### 1. Register for 
https://www....../ui/mall/

### 2. Click the red “Console” button in the top-right corner.

### 3. Click “Service Navigation” → “Artificial Intelligence” (blue button)

This opens the interface required for AI Notebook.

https://www....../ui/console/index.html#/notebook

### 4. Click “Billing” → “Overview” in the top-right corner.

### 5. Click “Recharge” → “Alipay”

Choose the recharge amount based on the service you need. The example below is expected to cost less than 30 RMB (roughly 20 RMB in practice).

### 6. After recharging, return to the AI Notebook interface.

### 7. Click “Notebook” → “Create Notebook”, choose ....................................................., and set the accelerator count to 2. Use two 4090 GPUs. Each accelerator provides 24 GB of memory, which is suitable for beginner debugging.

### 8. Click “Development Image” → “Base Image” → “Framework Name: PyTorch” → “Framework Version: 2.6.0” → “Python Version: py3.12-ubuntu22.04” → “CUDA/DTK Version: cuda12.4”, then click the red “Create” button in the lower-right corner.

This automatically creates the environment and switches back to the Notebook interface, where you can use Jupyter Notebook directly or log in via VS Code through Remote SSH.

### 9. Click “Quick Tools” → “JupyterLab”.

This loads the interface. The system usually allows network access automatically; if not, contact support.

### 10. Click “root” → “Notebook” → “Python3”.

It is also recommended to click the “+” tab at the top center and open “Other” → “Terminal”.

Note: the container uses the `root` account by default.

In [ ]:
pip install --upgrade pip

If the command above fails, switch to the Aliyun mirror first:

In [ ]:
pip config set global.index-url https://mirrors.aliyun.com/pypi/simple/
pip install --upgrade pip

In [ ]:
pip install transformers accelerate peft bitsandbytes datasets trl scikit-learn pandas

## Step 13: Download the dataset

Use the Kaggle dataset `crowdflower/twitter-airline-sentiment` as the example case. Registration is required, but it is free.

https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment

Click “Download” → “Download dataset as zip”. The CSV file is about 3 MB. Its contents look roughly like this:

| tweet_id | sentiment | author | content |
|---|---|---|---|
| 1956967341 | empty | xoshayzers | @tiffanylue i know i was listenin to bad habit earlier and i started freakin at his part =[ |
| 1956967666 | sadness | wannamama | Layin n bed with a headache ughhhh...waitin on your call... |

This is a well-known dataset and can be replaced with any dataset you choose.

## Step 14: Unzip and place the CSV file

Unzip the archive directly and drag `twitter-airline-sentimentSentiment_Analysis.csv` into the same folder as the `.ipynb` file on the left side of Jupyter Notebook (`/root/`).

## Step 15: Download the LLM to `/root/private_data/DeepSeek7B`

To speed up downloads, switch the Hugging Face mirror to `https://hf-mirror.com`.

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Define local save directory for the 7B model
local_model_dir = "/root/private_data/DeepSeek7B"

# ✅ Correct Model Name (7Billion parameters)
model_name = "deepseek-ai/deepseek-llm-7b-base"

print(f"Loading model: {model_name} (Official size: 7B parameters)")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,   # bfloat16 is safe and memory efficient
    device_map="auto"             # automatic device placement
)

# Save locally
tokenizer.save_pretrained(local_model_dir)
model.save_pretrained(local_model_dir)
print(f"Model saved to {local_model_dir}")

Downloading after changing the mirror typically takes about 5–10 minutes.

## Step 16: Official tip on environment backup

Translated from the official notebook documentation.

### A. Save the environment when powering off / save an image

You can save the development environment when shutting down or use the “Save Image” feature to back up the environment. This helps keep the machine configuration consistent and supports later restarts, team collaboration setups, and environment reproduction on other platforms. The container image can be saved both when the instance is on and when it is off.

### B. Important size limit

To ensure the image runs correctly, the data size of a single-layer image must not exceed 15 GiB. The system checks the image size. If the limit is exceeded, you need to manually move files from the container environment into file storage.

### C. Find large files quickly

```bash
cd /
find . -path "./proc" -prune -o \
       -path "/root/private_data/*" -prune -o  # exclude personal files
       -path "/root/public_data/*" -prune -o   # exclude platform-shared files
       -path "/root/group_data/*" -prune -o    # exclude team-shared files
       -path "/public/*" -prune -o             # exclude shared storage files
       -path "/work/*" -prune -o               # exclude shared storage files
       -type f -exec du -h {} + | sort -hr | head -n 20  # show the top 20 largest files
```

### D. Move large files to permanent storage

```bash
mv /root/model_file /root/private_data/model_file
```

### E. Fix permissions after moving

Files moved this way may not be usable in file storage because the owner is `root`. You may need to adjust permissions in the current environment:

```bash
# Replace user_name with your actual compute username
chown user_name:user_name /root/private_data/model_file
```

## Step 17: Test the downloaded LLM

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

local_model_dir = "/root/private_data/DeepSeek7B"

tokenizer = AutoTokenizer.from_pretrained(local_model_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    local_model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Example prompt
input_text = "Explain quantum computing in simple terms"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The following code blocks, 18.b–25.b, are demonstration and explanation only. They are not the final production training code. Experienced practitioners can skip ahead and go directly to section 26.

## Step 18.b: Load the first 5,000 rows from the CSV file

For quick experimentation, you can reduce this to the first 50 rows.

In [ ]:
import pandas as pd

df = pd.read_csv('twitter-airline-sentimentSentiment_Analysis.csv')
first_50 = df.head(5000)
print(f"Loaded {len(first_50)} rows")
print(first_50[['tweet_id', 'sentiment', 'author', 'content']].head())

## Step 19.b: Prepare the data for the trainer

The instruction differs from the earlier version. Because the 7B model has higher DoF, the prompt is tuned specifically for better stability.

In [ ]:
# Define instruction
instruction = "Analyze the sentiment of the following tweet. Output exactly one word, with no punctuation or extra text:"

# Create a list of formatted texts
formatted_texts = []
for idx, row in first_50.iterrows():
    text = f"Instruction: {instruction}\nInput: {row['content']}\nOutput: {row['sentiment']}"
    formatted_texts.append(text)

# Convert to a Hugging Face Dataset
from datasets import Dataset
dataset = Dataset.from_dict({"text": formatted_texts})

print(dataset[0]['text'])

## Step 20.b: Inspect the transformer layer class name

The example class name is:

`class 'transformers.models.llama.modeling_llama.LlamaDecoderLayer'`

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

local_model_dir = "/root/private_data/DeepSeek7B"
model = AutoModelForCausalLM.from_pretrained(local_model_dir, torch_dtype=torch.bfloat16)
print(type(model.model.layers[0]))   # should output something like <class 'transformers.models.llama.modeling_llama.LlamaDecoderLayer'>

# Replace the class name in the code below accordingly.

PyTorch defaults to the `spawn` multiprocessing start method. If needed, you can enforce it with the following code. The multiprocessing start method must be set before importing libraries.

In [ ]:
import multiprocessing
multiprocessing.set_start_method('spawn', force=True)
import torch.multiprocessing as tmp
tmp.set_start_method('spawn', force=True)

# Now import torch and other libraries (this will be after start method is set)
import torch
import functools
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from accelerate import Accelerator, notebook_launcher
from accelerate.utils import FullyShardedDataParallelPlugin
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy

# Verify start method (optional)
print(f"Start method: {multiprocessing.get_start_method()}")

## Step 21.b: Configure LoRA and FSDP

Load the model prototype for training, then configure LoRA and FSDP. LoRA must be applied before FSDP. For demonstration purposes, this notebook uses bfloat16. The initialization of `Accelerator` should be handled carefully.

In [ ]:
# Configuration
local_model_dir = "/root/private_data/DeepSeek7B"
output_dir = "/root/private_data/DeepSeek7B_finetuned"
lora_r = 8
lora_alpha = 32
lora_dropout = 0.1
target_modules = ["q_proj", "v_proj"]
batch_size = 2
grad_accum = 4
learning_rate = 2e-4
num_epochs = 3
max_length = 512


# ------------------------------
# Load tokenizer
# ------------------------------
tokenizer = AutoTokenizer.from_pretrained(local_model_dir, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# ------------------------------
# Load base model (bfloat16)
# ------------------------------
model = AutoModelForCausalLM.from_pretrained(
    local_model_dir,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)


# --- Apply LoRA ---
lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=target_modules,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)




# --- Cast everything to bfloat16 ---
model = model.to(torch.bfloat16)




# --- Detect transformer layer class for FSDP auto-wrap ---
try:
    base_model = model.base_model.model
    layer_class = type(base_model.model.layers[0])
    print(f"Detected layer class: {layer_class}")
except:
    from transformers.models.llama.modeling_llama import LlamaDecoderLayer
    layer_class = LlamaDecoderLayer

auto_wrap_policy = functools.partial(
    transformer_auto_wrap_policy,
    transformer_layer_cls={layer_class}
)
fsdp_plugin = FullyShardedDataParallelPlugin(
    auto_wrap_policy=auto_wrap_policy,
    use_orig_params=True,
)




# --- Accelerator with FSDP ---
accelerator = Accelerator(fsdp_plugin=fsdp_plugin)
model = accelerator.prepare(model)

## Step 22.b: Verify that FSDP distributed training is enabled

In [ ]:
# After accelerator.prepare(model)
print("=== FSDP Status ===")
print(f"Number of processes: {accelerator.num_processes}")
print(f"Process index: {accelerator.process_index}")
print(f"Device: {accelerator.device}")

# Check if the model is wrapped with FSDP (it should be)
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
if isinstance(model, FSDP):
    print("✅ Model is wrapped with FSDP.")
    # Print some info about the FSDP wrapped modules
    for name, module in model.named_modules():
        if isinstance(module, FSDP):
            print(f"FSDP module: {name} with params: {sum(p.numel() for p in module.parameters())}")
else:
    print("⚠️ Model is not an FSDP instance – maybe it's wrapped inside a PEFT model?")
    # PeftModel might wrap the base model, so we need to check base_model
    if hasattr(model, 'base_model') and isinstance(model.base_model, FSDP):
        print("✅ Base model (inside PeftModel) is wrapped with FSDP.")
    else:
        print("❌ Model not wrapped with FSDP. Check your FSDP plugin.")

# Optional: print device of first few parameters to see sharding
print("\n=== Parameter device assignment (first 5 named parameters) ===")
for i, (name, param) in enumerate(model.named_parameters()):
    if param.device != accelerator.device:   # might be different for sharded params
        print(f"{name}: device={param.device}")
    if i >= 5:
        break

## Step 23.b: Define the tokenizer function for training samples

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,   # adjust as needed
        return_tensors=None,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1)   # optional split
train_dataset = tokenized_dataset["train"]
eval_dataset = tokenized_dataset["test"]

## Step 24.b: Define the training process and start fine-tuning

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./deepseek-lora-fsdp",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    bf16=True,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_num_workers=4,
    ddp_find_unused_parameters=False,
    eval_strategy="steps",       # <-- changed from evaluation_strategy
    eval_steps=500,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=None,
)

trainer.train()


# Launch on 2 GPUs (change num_processes to 3 if you have three)
# notebook_launcher(train_function, num_processes=2, args=())

## Step 25.b: Save the fine-tuned files

In [ ]:
model.save_pretrained("./deepseek-lora-fsdp-final")
tokenizer.save_pretrained("./deepseek-lora-fsdp-final")

## Step 26: Practical production code

In practice, steps 18.b–25.b should be consolidated into a single file named `train_deepseek7B_finetuned.py` for real execution. That file should live in the same folder as the `.ipynb` notebook. Below is the actual code for `train_deepseek7B_finetuned.py`.

In [ ]:
import torch
import functools
import pandas as pd
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from accelerate import Accelerator
from accelerate.utils import FullyShardedDataParallelPlugin
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from torch.utils.data import DataLoader
from tqdm import tqdm

# Disable tokenizer parallelism
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Configuration
local_model_dir = "/root/private_data/DeepSeek7B"
output_dir = "/root/private_data/DeepSeek7B_finetuned"
lora_r = 8
lora_alpha = 32
lora_dropout = 0.1
target_modules = ["q_proj", "v_proj"]
batch_size = 2
grad_accum = 4
learning_rate = 2e-4
num_epochs = 3
max_length = 512

def main():
    # --- Load dataset ---
    df = pd.read_csv('twitter-airline-sentimentSentiment_Analysis.csv')
    first_50 = df.head(5000)
    # instruction = "Analyze the sentiment of the following tweet:"
    instruction = "Analyze the sentiment of the following tweet. Output exactly one word, with no punctuation or extra text:"
    formatted_texts = []
    for _, row in first_50.iterrows():
        text = f"Instruction: {instruction}\nInput: {row['content']}\nOutput: {row['sentiment']}"
        formatted_texts.append(text)
    dataset = Dataset.from_dict({"text": formatted_texts})

    # --- Tokenizer ---
    tokenizer = AutoTokenizer.from_pretrained(local_model_dir, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    # --- Load base model (dtype bfloat16) ---
    model = AutoModelForCausalLM.from_pretrained(
        local_model_dir,
        dtype=torch.bfloat16,
        trust_remote_code=True,
    )

    # --- Apply LoRA ---
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=target_modules,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    # --- Cast everything to bfloat16 ---
    model = model.to(torch.bfloat16)

    # --- Detect transformer layer class for FSDP auto-wrap ---
    try:
        base_model = model.base_model.model
        layer_class = type(base_model.model.layers[0])
        print(f"Detected layer class: {layer_class}")
    except:
        from transformers.models.llama.modeling_llama import LlamaDecoderLayer
        layer_class = LlamaDecoderLayer

    auto_wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={layer_class}
    )
    fsdp_plugin = FullyShardedDataParallelPlugin(
        auto_wrap_policy=auto_wrap_policy,
        use_orig_params=True,
    )

    # --- Accelerator with FSDP ---
    accelerator = Accelerator(fsdp_plugin=fsdp_plugin)
    model = accelerator.prepare(model)

    # --- Print trainable parameters (only on main process) ---
    if accelerator.is_main_process:
        model.print_trainable_parameters()
        print("=== FSDP Status ===")
        print(f"Number of processes: {accelerator.num_processes}")

        # Check embedding weight shape before training
        # We need to get the original embedding layer (the base model's embed_tokens)
        # Because the model is wrapped, we have to dig into the FSDP-wrapped structure.
        try:
            embed_layer = model.base_model.model.model.embed_tokens
            print(f"Embedding weight shape before training: {embed_layer.weight.shape}")
        except Exception as e:
            print(f"Could not inspect embedding weight: {e}")

    # --- Tokenize dataset ---
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
    tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1)
    train_dataset = tokenized_dataset["train"]
    eval_dataset = tokenized_dataset["test"]

    # --- Create data loader (num_workers=0 to avoid forking) ---
    def collate_fn(batch):
        # batch is a list of dicts with 'input_ids', 'attention_mask'
        input_ids = torch.stack([torch.tensor(item['input_ids']) for item in batch])
        attention_mask = torch.stack([torch.tensor(item['attention_mask']) for item in batch])
        labels = input_ids.clone()
        # Set padding tokens to -100 so they are ignored in loss
        labels[labels == tokenizer.pad_token_id] = -100
        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,   # crucial: no extra processes
    )
    eval_loader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
    )

    # --- Prepare optimizer and scheduler ---
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    # Use accelerator to prepare data loaders and optimizer (model already prepared)
    train_loader, eval_loader, optimizer = accelerator.prepare(train_loader, eval_loader, optimizer)

    # --- Training loop ---
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}", disable=not accelerator.is_main_process)
        for step, batch in enumerate(progress_bar):
            # Forward pass
            outputs = model(**batch)
            loss = outputs.loss
            loss = loss / grad_accum
            accelerator.backward(loss)

            if (step + 1) % grad_accum == 0 or step == len(train_loader) - 1:
                optimizer.step()
                optimizer.zero_grad()

            total_loss += loss.item() * grad_accum
            if accelerator.is_main_process:
                progress_bar.set_postfix({"loss": total_loss / (step + 1)})

        # Evaluation (optional)
        model.eval()
        eval_loss = 0
        with torch.no_grad():
            for batch in eval_loader:
                outputs = model(**batch)
                eval_loss += outputs.loss.item()
        eval_loss /= len(eval_loader)
        if accelerator.is_main_process:
            print(f"Epoch {epoch+1} - Train loss: {total_loss/len(train_loader):.4f}, Eval loss: {eval_loss:.4f}")
        model.train()

    # --- Save final model ---
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained("/root/private_data/DeepSeek7B_finetuned")
    tokenizer.save_pretrained("/root/private_data/DeepSeek7B_finetuned")
    print("Training completed and model saved.")

if __name__ == "__main__":
    main()

## Step 27: Restart the `.ipynb` kernel and run the following in a new cell

First, stop any existing `torchrun` processes:

In [ ]:
!pkill -f torchrun   # kills all torchrun processes

After clearing the GPU runtime, run the following command in another new cell to start training.

`--nproc_per_node=2` means two GPUs.

In [ ]:
# Run the command to train the network
!torchrun --nproc_per_node=2 --master_addr=127.0.0.1 --master_port=29501 train_deepseek7B_finetuned.py

With 5,000 training examples and 3 epochs, the default runtime is about 1.5 hours. For faster debugging, reduce the sample count.

You can monitor GPU utilization with the following command in the terminal:

```bash
watch -n 1 nvidia-smi
```

Both GPUs should be active. One GPU may use about 23 GB of memory and the other about 21 GB.

The fine-tuned model will be saved in `/root/private_data/DeepSeek7B_finetuned`.

## Step 28: Compare the fine-tuned model with the training dataset

In [ ]:
from peft import PeftModel
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# Paths
base_model_dir = "/root/private_data/DeepSeek7B"
lora_adapter_dir = "/root/private_data/DeepSeek7B_finetuned"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_dir, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load base model (use same dtype as during training)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto"          # automatically distributes layers across available GPUs/CPU
)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, lora_adapter_dir)
model.eval()

# Load dataset
df = pd.read_csv('twitter-airline-sentimentSentiment_Analysis.csv')   # adjust path if needed

# Randomly select a few examples
import random
test_indices = random.sample(range(len(df)), 5)

for idx in test_indices:
    tweet = df.iloc[idx]['content']
    true_sentiment = df.iloc[idx]['sentiment']
    prompt = f"Instruction: Analyze the sentiment of the following tweet. Output exactly one word, with no punctuation or extra text:\nInput: {tweet}\nOutput:"

    # Tokenize and move to the model's device
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate with no gradients (faster, less memory)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract the generated sentiment (text after "Output:")
    predicted = generated.split("Output:")[-1].strip().split("\n")[0]

    print(f"Tweet: {tweet[:80]}...")
    print(f"True sentiment: {true_sentiment}")
    print(f"Predicted: {predicted}")
    print("-" * 50)

## Step 29: Return to “Console” → “Notebook” → “Operations” and shut down the container to avoid extra charges.

## Step 30: Download the fine-tuned model

In the left-side “AI” → “File Management” area, you can download the fine-tuned model from `/root/private_data/DeepSeek7B_finetuned`.

## Conclusion

At this point, the 7B fine-tuned model has been completed using multi-GPU FSDP + LoRA distributed training.

This type of fine-tuned model can be deployed locally with high efficiency, significantly reducing API latency and cost. Compared with smaller models, a larger model with billions to tens of billions of parameters has higher freedom and can serve as the backbone for agentic AI agents or specialized tool-oriented models. It is also suitable for professional environments that need local deployment but face hardware constraints.

The same approach can be extended directly to 10B-class models within about 600 GB of storage, using 8 A800 GPUs with 80 GB memory per GPU. With more GPUs and 4-bit quantization, the method can scale to models from tens of billions to hundreds of billions of parameters, covering the needs of most first-tier, second-tier, and small-to-medium enterprises outside a few top-tier companies.

Contact for job opportunities or project collaboration: `yucongcai_business@outlook.com`

For research-related matters: `yucongcai_research@outlook.com`